# Graph Construction & Visualization — CIC-IDS-2017 Flow Logs

**Companion notebook to `01_graph_construction_and_visualization_kaggle.ipynb`.**

That notebook started from graphs someone else had *already built* (the Kaggle
`.graphml` files). This notebook does the step before that: it starts from raw
**labeled network flow logs** (CIC-IDS-2017, `.csv` files where each row is one
network flow between two IP addresses) and builds the communication graph
ourselves, from scratch, so every design decision is visible in the code.

**What a "communication graph" means here**

| Graph element | Built from |
|---|---|
| Node | An IP address that appears as a Source IP or Destination IP in at least one flow |
| Edge | An undirected connection between two IP addresses that exchanged **at least one** flow |
| Edge weight `flow_count` | How many separate flows (rows) occurred between that pair |
| Edge weight `total_packets`, `total_bytes` | Summed traffic volume between that pair |
| Edge weight `attack_flows` | How many of those flows were labeled as an attack (not `BENIGN`) |
| Node attribute `is_malicious` | `True` if the IP touched *any* attack-labeled flow |

**Scope and honesty about sampling.** The real CIC-IDS-2017 files are large
(the Friday DDoS file alone has 225,746 rows) and — this matters — the rows
are stored **chronologically**, so attacks are concentrated later in the file,
not spread throughout it. Naively reading the first `N` rows of an attack file
finds zero attacks. To avoid that trap, this notebook uses
`sample_labeled_flows()` (`src/graph_construction.py`), which scans the file in
chunks until it has found the requested number of benign and attack rows,
regardless of where in the file they occur. For classroom purposes we work
with a modest number of real, verified rows per file — enough to build a
graph you can read on one slide and trace back to real data, not a
comprehensive statistical sample of the full file.

**Datasets used in this notebook** (`data/cic-ids-2017/GeneratedLabelledFlows/`):
- `Monday-WorkingHours.pcap_ISCX.csv` — 100% benign, the "normal day" baseline
- `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv` — DDoS attack
- `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv` — port scan attack
- `Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv` — web application attacks (SQLi/XSS/brute force)
- `Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv` — infiltration attack

In [ ]:
# Cross-platform, kernel-safe install: sys.executable always points at the
# exact Python this notebook's kernel is running under - the same code
# installs correctly on Windows, macOS, and Linux, and inside any venv/
# conda environment, unlike a hardcoded interpreter path.
import sys
!{sys.executable} -m pip install -r ../requirements.txt

In [ ]:
from __future__ import annotations  # Must be line 1!

import sys
sys.path.insert(0, "../src")

import pandas as pd

from graph_construction import (
    load_flow_logs,
    sample_labeled_flows,
    build_communication_graph,
    build_windowed_graphs,
    graph_summary,
)
from graph_features import compute_all_features, verify_features
from visualize import plot_graph, plot_multiple_graphs

%matplotlib inline

In [ ]:
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)

DATA_DIR = "../data/cic-ids-2017/GeneratedLabelledFlows"

## Section 1 — Benign baseline: an ordinary Monday

`Monday-WorkingHours.pcap_ISCX.csv` contains no attacks at all — every flow is
labeled `BENIGN`. We use it as the control graph: what does *normal* office
network traffic look like before we go looking for attacks?

In [ ]:
monday_path = f"{DATA_DIR}/Monday-WorkingHours.pcap_ISCX.csv"

# Monday is 100% benign, so we just ask for benign rows. sample_labeled_flows
# scans in chunks rather than assuming the first N rows are representative -
# a habit worth keeping even on a file where it happens not to matter, since
# it is the same function we rely on for the attack files below.
monday_flows = sample_labeled_flows(monday_path, n=15, want="benign", chunksize=2000)
print(f"Loaded {len(monday_flows)} real benign flows from {monday_path.split('/')[-1]}")

G_monday = build_communication_graph(monday_flows)
print("Graph summary:", graph_summary(G_monday))

plot_graph(G_monday, title="Monday — benign baseline", save_path="figures/fig_01_monday_baseline.png");

**Reading this graph.** Even ordinary traffic naturally forms a small,
sparse graph: most hosts talk to exactly one other host (an NTP server, a DNS
server, a website), while a couple of internal hosts (like
`192.168.10.17`, which polls several NTP servers) have a slightly higher
degree. Nothing here is colored red, because nothing is labeled an attack —
this is the "quiet" shape we will compare the attack graphs against.

## Section 2 — Attack-day graphs

For each attack file we pull a handful of real benign flows (for context) and
a handful of real attack-labeled flows (using the fixed, chunk-scanning
`sample_labeled_flows`, since these files are chronologically ordered and the
attacks do not start at row 1), combine them, and build one graph per file.

In [ ]:
attack_files = {
    "DDoS (Friday)": f"{DATA_DIR}/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "PortScan (Friday)": f"{DATA_DIR}/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Web Attack (Thursday AM)": f"{DATA_DIR}/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Infiltration (Thursday PM)": f"{DATA_DIR}/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
}

attack_graphs = {}
for name, path in attack_files.items():
    benign_rows = sample_labeled_flows(path, n=10, want="benign", chunksize=2000)
    attack_rows = sample_labeled_flows(path, n=10, want="attack", chunksize=2000)
    combined = pd.concat([benign_rows, attack_rows], ignore_index=True)

    G = build_communication_graph(combined)
    attack_graphs[name] = G

    print(f"{name}: {len(benign_rows)} benign + {len(attack_rows)} attack real flows "
          f"-> {graph_summary(G)}")

In [ ]:
plot_multiple_graphs(
    attack_graphs,
    max_nodes=30,
    ncols=2,
    suptitle="Attack-day communication graphs (red = touched an attack flow)",
    save_path="figures/fig_02_attack_day_graphs.png",
);

**First read of the four panels.** In every attack graph, exactly one
or two nodes are colored red (attack-labeled), and they sit right where you'd
expect: `172.16.0.1` (the attacking machine, external to the `192.168.10.0/24`
office subnet) and `192.168.10.50` (the target server) show up as the
red-flagged pair in the DDoS and PortScan graphs. The rest of the graph — the
benign context flows — is untouched and colored blue, exactly as in the Monday
baseline.

## Section 3 — A key finding: many flows can collapse into *one* edge

Look closely at the DDoS and PortScan summaries above: both attacks produced
graphs with **very few edges**, not many. Let's print the actual edges to see
why.

In [ ]:
for name in ["DDoS (Friday)", "PortScan (Friday)"]:
    G = attack_graphs[name]
    print(f"--- {name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges ---")
    for u, v, d in G.edges(data=True):
        flag = "  <-- attack edge" if d["attack_flows"] > 0 else ""
        print(f"  {u:>16} -- {v:<16} flow_count={d['flow_count']:>3} "
            f"attack_flows={d['attack_flows']:>3}{flag}")
    print()

**Why this matters.** `172.16.0.1` and `192.168.10.50` exchanged 10
separate flows in our sample — but every one of those flows was between the
*same two IP addresses*, just on different destination ports (that is
literally what a port scan is: one attacker, one target, many ports). Because
our graph's edges are defined at the **IP-pair level**, all 10 of those flows
collapse into a single edge. The same thing happens with the DDoS sample: many
flows, same two endpoints, one edge.

This is not a bug — it is an honest limitation of IP-level communication
graphs that is worth stating out loud in a presentation: **a simple IP graph
cannot visually distinguish "one heavy but legitimate conversation" from "one
port scan" from "one DDoS burst" by topology alone**, because all three look
like a single edge between two nodes. The signal that separates them lives in
the edge *weights* (`flow_count`, `attack_flows`, and — if you look at the raw
columns — the fact that the destination port changes on every flow), not in
the shape of the graph.

**Common mistake to avoid:** assuming "more edges/more spread-out graph = more
suspicious." Here it is the *opposite* — the attack graphs have *fewer* edges
than the benign Monday baseline, because the attacker talks to one target
repeatedly instead of many peers occasionally. Degree and edge count alone are
not attack indicators; they have to be read together with edge weights and
labels.

## Section 4 — Time-window graphs: watching the network over time

So far each graph has treated its sample as one static snapshot. But every
flow has a real timestamp, and `build_windowed_graphs()` floors each
timestamp into fixed-size bins (here, 5-minute windows) and builds one graph
per bin — letting us see the network evolve rather than freezing it into a
single picture.

In [ ]:
monday_flows_ts = load_flow_logs(monday_path)  # need the full Timestamp column, not just the label-sampled subset
monday_flows_ts = monday_flows_ts[monday_flows_ts["Timestamp"].notna()]
print("Timestamp range in this sample:", monday_flows_ts["Timestamp"].min(), "to", monday_flows_ts["Timestamp"].max())

windowed = build_windowed_graphs(monday_flows_ts, window="5min")
print(f"{len(windowed)} time windows found")
for window_start, G in windowed.items():
    print(f"  {window_start}: {graph_summary(G)}")

plot_multiple_graphs(
    {str(k): v for k, v in windowed.items()},
    max_nodes=30,
    ncols=2,
    suptitle="Monday traffic split into 5-minute windows",
    save_path="figures/fig_04_time_windows.png",
);

**Reading this.** The two windows are genuinely different snapshots of
the same day (08:55 and 09:00), each with its own hosts and edges. In a full,
multi-hour capture this is what lets you see traffic patterns build up,
persist, or change abruptly over time — including the moment an attack
starts, which is exactly the transition point you would zoom in on if you
were investigating an incident rather than looking at a whole day at once.

## Section 5 — Feature extraction and verification

The centrality formulas (degree, closeness, betweenness, PageRank, clustering)
are the same ones used in notebook 1 — see that notebook for the full
definitions. Here we just confirm the same pipeline (`compute_all_features`,
`verify_features` from `src/graph_features.py`) runs correctly on a graph we
built ourselves, and that the malicious flag lines up with what we already
know from Section 2.

In [ ]:
G = attack_graphs["DDoS (Friday)"]
features = compute_all_features(G)
print(features.round(4))
print()

checks = verify_features(G, features)
print("Verification checks:")
for k, v in checks.items():
    print(f"  {k}: {v}")
assert checks["sum_of_degrees_equals_2x_edges"], "Handshake lemma failed"
assert checks["pagerank_sums_to_approximately_1"], "PageRank does not sum to ~1"
print("\nBoth verification checks passed.")

**Interpretation.** In this small 8-node sample, `172.16.0.1` and
`192.168.10.50` — the two nodes marked `is_malicious = True` — do not
necessarily have the highest centrality scores; with so few nodes the
centrality numbers mostly reflect this tiny sample's shape, not a
statistically meaningful ranking. That is expected and worth saying plainly
in a presentation: this notebook demonstrates the *mechanics* of the pipeline
(load real flows → build a real graph → extract real, verified features) on a
small, traceable sample. Running the identical code against the full,
multi-hundred-thousand-row files (or against the large pre-built Kaggle graphs
in notebook 1, which give the large-scale numeric picture: 41,073 nodes, one
hub carrying 30% of all edges, 670 attack-touched nodes) is where centrality
rankings become statistically meaningful.

## Summary — talking points for the presentation

- **Pipeline demonstrated end-to-end, from raw data:** flow-log CSV → labeled
  DataFrame → weighted undirected graph → centrality features, with a
  verification check at every stage (handshake lemma, PageRank sums to ~1).
- **Real bug found and fixed along the way:** naively reading the first `N`
  rows of an attack file (e.g. `nrows=200`) found **zero** attack flows,
  because CIC-IDS-2017 files are stored chronologically and attacks are
  concentrated later in the file. `sample_labeled_flows()` fixes this by
  scanning in chunks until it finds real examples of the requested label,
  wherever they occur.
- **Benign baseline (Monday):** normal traffic is naturally sparse — most
  hosts talk to exactly one peer.
- **Attack graphs (DDoS, PortScan, Web Attack, Infiltration):** each
  correctly isolates the attacking IP and target IP as the only
  `is_malicious` nodes, using real, verified flow data from each file.
- **Key structural finding:** DDoS and port-scan traffic can collapse onto a
  *single* graph edge, because many flows between the same two IPs merge into
  one edge under this graph definition. Topology alone cannot tell these
  attacks apart from a single busy conversation — the tell is in the edge
  weights (`flow_count`, `attack_flows`), not the shape of the graph.
- **Time matters:** `build_windowed_graphs()` shows the same day as a sequence
  of graphs rather than one frozen snapshot, which is the natural next step
  toward watching an attack unfold over time.
- **Companion to notebook 1:** this notebook builds graphs *from scratch* out
  of raw flow logs (small, traceable samples); notebook 1 analyzes a much
  larger, pre-built graph (41k+ nodes) to show what these same metrics look
  like at real scale, and additionally covers motif counts, Node2Vec
  embeddings, and Random Forest / XGBoost / GCN / GraphSAGE / GAT model
  training - not repeated here since this notebook's samples are
  intentionally too small for any of that to be statistically meaningful.